<a href="https://colab.research.google.com/github/GabCAD92/Machine-learning-tasks/blob/main/Gabriel_Owolabi_Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Supervised Machine Learning**
## **Supervised Learning: Transformers practice**

This lesson implements **Encoder-only** and **Decoder-Only Transformer** architectures and discusses **Encoder-Decoder Transformers** in the context of developing an **automotive troubleshooting assistant**.

### Professor:

<img src="https://www.sorocaba.unesp.br/Home/Graduacao/EngenhariadeControleeAutomacao/alexandre/alex_marta1_small.jpg" width="100" style="float: left; margin-right: 5px;" border="10px" />

  __Prof. Dr. Alexandre da Silva Simões__ <br>
  Control and Automation Engineering Department (DECA) <br>
  Institute of Science and Technology<br>
  São Paulo State University (Unesp) <br>
  Campus Sorocaba <br>
  www.sorocaba.unesp.br/professor/assimoes

<br/>


____

# **Table of Contents**

**PART I - Encoder-Only Transformeres**

1. Introduction <br>
  1.1. Automotive Diagnostic System <br>
  1.2. Encoder-only Transformer <br>
  1.3. Bidirectional Encoder Representations from Transformers (BERT) <br>
2. Installing Packages <br>
3. Importing Librariers <br>
4. Pre-trained Sentence-BERT <br>
5. **Scenario I: Semantic Search** <br>
   5.1. Building a small Knowledge Base <br>
   5.2. Generating the Embeddings <br>
   5.3. Semantic Search <br>
   5.4. Constructing Queries <br>
   5.5. Keyword-based search <br>
   5.6. Didactic comparison <br>
   5.7. Building a simple interface <br>
6. **Scenario II: Retrieving multiple chunks** <br>
  6.1. Installing libraries <br>
  6.2. Loading/Reusing BERT<br>
  6.3. Selecting the Knowledge Base<br>
  6.4. Chunks generation<br>
  6.5. Embedding Generation<br>
  6.6. Semantic Retrieval<br>
7. **Scenario III: BERT Fine-tuning** <br>
  7.1. Intalling libraries<br>
  7.2. Loading/Reusing BERT<br>
  7.3. Downloading/reusing manual and chunks<br>
  7.4. Preparing the training<br>
  7.5. Training<br>
  7.6. Recreating embeddings<br>
  7.7. Comparing models<br>
8 . Semantic search vs fine-tuning approach<br>
9. Other Encoder-only Transformers <br>

**PART II - Decoder-Only Transformers**

11. Qwen <br>
  11.1. Loading Qwen <br>
  11.2. Prompt construction <br>
  11.3. Retrieval-Augmented Generation (RAG) <br>
  11.4. First prompt inspection<br>
  11.5. Runing the RAG <br>
12. Encoder-only vs Decoder-Only Transformers <br>
  12.1. Overview <br>
  12.2. Fine-Tuning <br>

**PART III - Encoder-Decoder Transformers**

13. Overview <br>
14. Available models <br>
15. Next steps...

<br>

____



# **1. Introduction**


## **1.1. Task: Automotive Troubleshooting Assistant**

<div align="center">
    <img src="https://drive.google.com/uc?export=view&id=1hl2-JpbziM7Y086Sz2Wr9H7izlFJ0Jji" width="500">
</div>

An **Automotive Troubleshooting Assistant** is an AI-powered system designed to help drivers, technicians, and maintenance professionals identify, analyze, and resolve vehicle problems more efficiently. By processing symptom descriptions, technical manuals, maintenance records, and even images of vehicle components, the assistant can suggest likely causes of failures, retrieve relevant technical information, recommend diagnostic procedures, and support decision-making during vehicle maintenance.



## **1.2. Encoder-only Transformer**

<center>

![Imagem](https://drive.google.com/uc?export=view&id=1KzaeRId-K0MobeMh_bybn921i9qcQsGC)

</center>

Main encoder-only Transformer applications:

- **Text Classification**: Identify categories, sentiments, topics;
- **Information Retrieval**: Find relevant documents;
- **Semantic Search**: Measure similarity between texts;
- **Question Answering**: Find answers within a context;
- **Entity Recognition**: Detect names, locations, organizations, etc.;
- **Embeddings**: Generate vector representations for downstream tasks;
- **Image Classification**: Recognize objects or scenes;
- **Visual Segmentation and Detection**: Serve as a backbone for computer vision tasks;
- **RAG**: Encode queries and documents
- **Recommendation**: Represent users and items in vector spaces.








-----

## **1.3. Bidirectional Encoder Representations from Transformers (BERT)**

### **1.3.1. What is BERT?**

**BERT (*Bidirectional Encoder Representations from Transformers*)** is a language model introduced by Google in 2018 that uses only the encoder part of the Transformer architecture. The main characteristic of BERT is that **it reads a sentence in a bidirectional way**, meaning it simultaneously considers the words to the left and right of each term in order to understand its meaning in context.

### **1.3.2. How is BERT trained?**

Originally, BERT is trained in a **self-supervised** way using two main strategies:

1. Masked Language Modeling (MLM). Some words are hidden: "*The car [MASK] is discharged*.". The model learns to predict: "*battery*"

2. Next Sentence Prediction (NSP). The model learns relationships between consecutive sentences.

### **1.3.3. Potentialities and limitations**

* The BERT **pipeline** can be expressed by: *Text → Tokenization → Embeddings → Transformer Encoder → Contextual Representation*

* The output is a numerical vector (**embedding**) that represents the meaning of the text. In other words, based on the input: "*The car misfires when I accelerate, and the check engine light turns on*.", BERT could generate a vector representation such as: [0.12, -0.45, 0.87, ...]

* BERT was **not designed to generate free-form text** like encoder-decoder Transformers.
<br>

### **1.3.4. Overview in the Automotive Diagnostic System domain**
<br>

![Imagem](https://drive.google.com/uc?export=view&id=1GxQIvXUVLzbQLqtD02gpbva6EVy1tBIY)




# 2. **Installing Packages**

First of all, let's install the **sentence-transformers** library, which is built on top of *PyTorch* and *Hugging Face Transformers*.

In [1]:
# ============================================================
# Install
# ============================================================

!pip install -q sentence-transformers scikit-learn pandas

# **3. Importing Librariers**

Now, let's load the some **libraries** that will be responsable for:

- transforming: text → vectorial embedding (SentenceTransformer)
- Measuring how semantically similar two texts are (cosine_similarity)

In [2]:
# ============================================================
# Imports
# ============================================================

import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# **4. Pre-trained Sentence-BERT**

Now let's load our pre-trained BERT...

In [3]:
# ============================================================
# Load pre-trained BERT sentence
# ============================================================

model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

We downloaded a **pre-trained Sentence-BERT** that has already learned:
- syntax;
- semantics;
- linguistic relations.

The model selected is the **paraphrase-multilingual-MiniLM-L12-v2** <br/>
Let's check some of its main features...

### **1. Paraphrase:**

Indicates that the model was trained to **detect semantic similarity**, especially between semantically equivalent phrases. Example: <br>
*“O carro está falhando para iniciar*” <br>
and: <br>
“*O motor apresenta perda de combustão*”<br>
should generate similar embeddings.<br>

This is essential for semantic search, retrieval, RAG, matching FAQ.<br>
Some highlights:
- Embedding is not interpretable individually dimension by dimension;
- The meaning is distributed throughout the entire vector!



In [4]:
# ============================================================
# Show sentence embeddings generated for similar phrases
# ============================================================

docs = [
    { "texto": "O carro está falhando para iniciar."},
    { "texto": "O motor apresenta perda de combustão."},
    { "texto": "A pintura está manchada."},
]
df = pd.DataFrame(docs)

document_texts = df["texto"].tolist()

doc_embeddings = model.encode(
    document_texts,
    convert_to_tensor=False,
    normalize_embeddings=True
)

print(doc_embeddings.shape)

df2 = pd.DataFrame(doc_embeddings)

vmin = df2.min().min()
vmax = df2.max().max()
vmin
df2.style \
    .background_gradient(
        cmap="Blues",
        vmin=vmin,
        vmax=vmax
    ) \
    .format("{:.3f}")

(3, 384)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262,263,264,265,266,267,268,269,270,271,272,273,274,275,276,277,278,279,280,281,282,283,284,285,286,287,288,289,290,291,292,293,294,295,296,297,298,299,300,301,302,303,304,305,306,307,308,309,310,311,312,313,314,315,316,317,318,319,320,321,322,323,324,325,326,327,328,329,330,331,332,333,334,335,336,337,338,339,340,341,342,343,344,345,346,347,348,349,350,351,352,353,354,355,356,357,358,359,360,361,362,363,364,365,366,367,368,369,370,371,372,373,374,375,376,377,378,379,380,381,382,383
0,0.030,-0.030,0.032,0.037,0.041,-0.015,-0.036,0.008,-0.022,-0.084,-0.014,0.033,-0.050,-0.020,-0.069,0.080,-0.012,-0.005,0.030,-0.038,-0.040,-0.023,-0.007,0.042,0.015,0.075,0.010,0.059,-0.028,-0.004,-0.051,-0.023,0.055,0.078,0.045,-0.072,-0.018,0.003,0.000,-0.035,-0.017,-0.002,-0.032,-0.056,0.109,0.065,0.082,0.115,0.076,-0.113,-0.042,0.077,-0.023,-0.128,-0.063,-0.023,0.056,-0.003,-0.017,0.026,0.008,-0.097,-0.035,-0.040,0.065,-0.009,-0.000,-0.001,0.016,0.095,-0.019,0.029,-0.087,-0.060,-0.046,0.009,0.026,0.017,0.034,-0.043,0.045,-0.024,-0.019,0.028,-0.004,-0.008,0.020,-0.013,-0.039,-0.022,-0.027,-0.043,0.017,0.021,0.099,0.006,0.115,-0.019,-0.009,0.062,0.021,0.011,0.049,0.002,-0.040,0.033,-0.024,-0.016,-0.075,-0.008,0.051,-0.017,-0.023,0.079,-0.023,0.008,-0.070,-0.022,-0.086,0.065,-0.056,-0.079,0.030,0.050,-0.009,-0.059,-0.027,-0.002,0.006,-0.045,-0.064,0.072,-0.014,-0.016,-0.028,0.090,0.037,0.054,0.028,-0.128,-0.086,-0.110,-0.020,-0.047,-0.011,-0.042,-0.100,-0.058,0.058,-0.020,0.041,0.003,0.009,0.121,-0.070,0.056,-0.051,0.008,-0.002,0.065,0.028,0.021,0.044,-0.026,0.025,0.047,0.011,-0.046,0.032,0.041,-0.029,-0.086,0.017,-0.007,0.041,0.052,0.031,0.037,-0.051,0.010,-0.025,0.028,-0.076,0.021,0.085,0.045,-0.108,-0.050,-0.029,-0.042,-0.008,-0.057,0.056,0.001,-0.013,0.046,-0.055,0.021,0.054,-0.031,0.007,0.051,0.015,-0.006,0.010,-0.022,-0.021,-0.048,0.007,0.025,0.074,-0.083,0.067,-0.052,0.023,-0.011,-0.062,0.020,0.033,0.046,-0.060,0.032,0.065,0.017,0.090,-0.007,-0.029,-0.057,0.017,0.029,0.029,-0.044,-0.043,0.024,-0.071,0.017,0.013,0.052,0.035,-0.048,0.005,-0.019,-0.031,-0.027,0.022,-0.063,-0.048,-0.019,0.020,0.012,-0.025,0.035,-0.032,0.027,0.053,0.002,0.095,0.024,0.015,0.031,0.108,-0.030,-0.089,-0.110,0.045,0.030,0.007,-0.007,0.063,0.006,0.025,-0.101,0.007,0.037,0.118,-0.047,0.033,0.126,0.014,0.016,0.018,0.091,-0.055,0.083,0.052,0.036,-0.012,-0.070,0.016,-0.066,-0.000,0.055,0.059,-0.073,-0.010,-0.013,0.016,-0.026,0.012,0.073,-0.006,-0.015,0.079,-0.035,0.007,-0.086,0.013,0.083,-0.011,-0.003,0.005,0.025,0.036,0.075,0.013,-0.001,-0.041,0.127,-0.087,-0.035,0.018,-0.010,-0.040,-0.023,0.080,-0.064,-0.022,-0.026,-0.115,-0.079,-0.096,-0.047,0.016,0.000,-0.034,-0.046,-0.036,0.007,-0.008,-0.048,0.030,0.038,-0.046,0.035,0.061,-0.084,0.022,-0.061,0.061,0.060,0.019,0.017,0.070,0.016,-0.011,0.027,0.006,0.122,0.111,-0.053,0.083,0.047,-0.079,-0.036,0.037,0.087,-0.091,-0.055,0.012,-0.117,0.042,-0.030,-0.095,0.043,0.026,-0.070,-0.111,0.092,0.003,-0.054,-0.015,0.019,-0.037,0.042
1,0.000,0.044,-0.004,0.073,0.039,0.028,-0.009,0.057,0.024,-0.125,-0.020,0.063,

The `cosine_similarity()` function calculates the **similarity between two vector embeddings**: <br>
- Geometrically, it measures the angle between the vectors in multidimensional space: the smaller the angle, the greater the semantic similarity between the documents. <br>
- Since the embeddings have been **normalized** (`normalize_embeddings=True`), the comparison depends mainly on the direction of the vectors. Values ​​close to 1 indicate semantically similar texts, even if they use different words.

In [5]:
# ============================================================
# How similar are the embeddings?
# ============================================================

from sklearn.metrics.pairwise import cosine_similarity

cs1 = cosine_similarity(
    [doc_embeddings[0]],
    [doc_embeddings[1]]
)[0,0]

cs2 = cosine_similarity(
    [doc_embeddings[0]],
    [doc_embeddings[2]]
)[0,0]

cs3 = cosine_similarity(
    [doc_embeddings[1]],
    [doc_embeddings[2]]
)[0,0]

print("Similarity between: \n0 and 1: ", cs1, "\n0 and 2: ", cs2, "\n1 and 2: ", cs3)

Similarity between: 
0 and 1:  0.6219971 
0 and 2:  0.21351615 
1 and 2:  0.22889528


### **2. Multilingual**

The model supports 50+ languages, inclusing: Portuguese, English, Spanish, French, German, etc. This is VERY important because **semantically equivalent sentences in different languages are located close together in the vector space**!

Example:
- Portuguese: “*O carro não liga*”
- English: “*The car does not start*” <br/>
The embeddings tend to be close together.

In [6]:
# ============================================================
# Show embeddings for phrases in distinct languages
# ============================================================

docs = [
    { "texto": "O carro não liga."},
    { "texto": "The car does not start."},
    { "texto": "A pintura está manchada."},
]
df = pd.DataFrame(docs)

document_texts = df["texto"].tolist()

doc_embeddings = model.encode(
    document_texts,
    convert_to_tensor=False,
    normalize_embeddings=True
)

print(doc_embeddings.shape)

df2 = pd.DataFrame(doc_embeddings)

vmin = df2.min().min()
vmax = df2.max().max()
vmin
df2.style \
    .background_gradient(
        cmap="Blues",
        vmin=vmin,
        vmax=vmax
    ) \
    .format("{:.3f}")

(3, 384)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262,263,264,265,266,267,268,269,270,271,272,273,274,275,276,277,278,279,280,281,282,283,284,285,286,287,288,289,290,291,292,293,294,295,296,297,298,299,300,301,302,303,304,305,306,307,308,309,310,311,312,313,314,315,316,317,318,319,320,321,322,323,324,325,326,327,328,329,330,331,332,333,334,335,336,337,338,339,340,341,342,343,344,345,346,347,348,349,350,351,352,353,354,355,356,357,358,359,360,361,362,363,364,365,366,367,368,369,370,371,372,373,374,375,376,377,378,379,380,381,382,383
0,0.044,0.025,0.080,0.020,0.055,-0.027,0.019,0.024,-0.023,-0.095,0.026,0.024,-0.067,-0.019,-0.047,0.086,0.058,0.028,-0.003,-0.052,-0.034,-0.008,-0.001,0.068,0.011,0.058,-0.046,0.063,-0.049,-0.001,-0.052,0.018,0.014,0.068,0.036,-0.069,-0.013,0.015,-0.015,-0.079,-0.027,-0.035,-0.027,-0.027,0.106,0.044,0.081,0.108,0.040,-0.079,0.005,0.059,-0.035,-0.045,-0.059,0.006,0.031,-0.009,-0.001,-0.014,-0.032,-0.060,-0.036,-0.017,0.017,0.002,-0.046,-0.020,-0.043,0.050,0.024,0.030,-0.084,-0.021,-0.015,-0.017,-0.012,0.005,0.070,-0.040,0.020,-0.010,-0.046,-0.012,-0.020,-0.020,-0.041,0.042,-0.076,-0.025,-0.013,-0.044,0.016,0.013,0.031,-0.004,0.116,0.020,-0.050,0.087,0.069,0.028,0.013,0.047,-0.023,0.036,-0.027,-0.052,-0.083,0.024,0.027,-0.029,-0.031,0.025,-0.097,-0.024,-0.043,0.040,-0.022,0.069,-0.073,-0.068,0.012,0.043,-0.025,-0.045,-0.013,0.019,-0.025,-0.032,-0.025,0.048,-0.001,-0.026,-0.040,0.076,0.064,0.108,-0.005,-0.127,-0.027,-0.120,0.008,0.013,-0.059,-0.057,-0.026,-0.066,0.056,0.005,-0.033,0.022,0.017,0.063,-0.093,0.041,-0.027,0.033,-0.011,0.067,0.037,0.011,0.017,-0.013,-0.037,0.027,0.004,-0.033,0.029,0.023,-0.030,-0.108,0.026,0.013,0.076,0.035,-0.017,0.059,-0.024,-0.024,-0.036,0.031,-0.110,0.051,0.068,0.037,-0.126,-0.055,-0.029,-0.077,0.053,-0.147,0.071,-0.024,-0.039,0.040,-0.033,-0.002,0.058,0.014,0.015,0.012,0.079,-0.036,0.003,-0.055,-0.013,-0.058,-0.017,0.052,0.092,-0.060,0.067,-0.019,0.020,-0.014,-0.030,0.028,-0.022,0.008,-0.051,0.025,0.073,-0.001,0.108,-0.043,-0.010,-0.010,-0.034,0.005,0.031,-0.031,-0.027,0.028,-0.057,0.016,0.003,0.061,0.110,-0.008,-0.026,-0.027,-0.065,0.033,-0.047,-0.064,-0.068,0.043,0.003,0.001,-0.056,0.069,0.023,0.042,0.062,0.017,0.086,0.061,-0.008,-0.016,0.092,0.008,-0.112,-0.098,0.000,0.034,-0.003,-0.026,0.076,-0.013,-0.030,-0.110,-0.022,-0.037,0.118,-0.072,0.040,0.117,0.023,0.053,0.024,0.074,-0.009,0.070,0.098,0.014,-0.001,-0.048,0.050,-0.047,0.018,0.080,0.100,-0.064,-0.006,0.006,-0.014,-0.049,0.056,0.127,0.067,0.011,0.042,-0.029,0.030,-0.047,-0.004,0.066,-0.088,-0.034,-0.040,0.027,0.061,0.057,0.039,0.061,-0.005,0.136,-0.063,-0.020,-0.006,-0.000,-0.064,-0.041,0.025,-0.076,0.016,-0.066,-0.084,-0.049,-0.076,-0.042,-0.045,0.009,-0.005,-0.069,-0.051,-0.017,0.002,0.002,0.049,0.044,-0.014,0.028,0.079,-0.059,-0.019,-0.069,0.076,0.045,0.006,0.039,0.028,0.061,-0.046,-0.011,0.018,0.062,0.106,-0.026,0.069,0.000,-0.067,-0.015,0.018,0.028,-0.046,-0.061,-0.017,-0.076,0.027,0.013,-0.084,0.073,-0.018,-0.132,-0.078,0.072,-0.003,-0.038,-0.003,0.038,-0.028,0.045
1,0.044,0.005,0.028,0.025,0.053,0.016,-0.008,0.054,0.005,-0.081,-0.0

In [7]:
# ============================================================
# How similar are the embeddings?
# ============================================================

from sklearn.metrics.pairwise import cosine_similarity

cs1 = cosine_similarity(
    [doc_embeddings[0]],
    [doc_embeddings[1]]
)[0,0]

cs2 = cosine_similarity(
    [doc_embeddings[0]],
    [doc_embeddings[2]]
)[0,0]

cs3 = cosine_similarity(
    [doc_embeddings[1]],
    [doc_embeddings[2]]
)[0,0]

print("Similarity between: \n0 and 1: ", cs1, "\n0 and 2: ", cs2, "\n1 and 2: ", cs3)

Similarity between: 
0 and 1:  0.85482657 
0 and 2:  0.22141737 
1 and 2:  0.19292738


### **3. MiniLM**

The model's basis is **MiniLM**, that is, a compact Transformer based on **knowledge distillation**. The central idea is training a small model to mimic the behavior of a larger model:
- Teacher model: Large and powerful Transformer
- Student model: Compact Transformer learning the teacher's patterns <br/>

Objective: reduce memory, latency, FLOPs, computational cost.
Result: We can maintain good semantic qualit with lower computational cost
This explains why it runs so well in Colab!

### **4. L12**

12 layers Transformer Architecture.
Comparison:

|Model    | Layers |
| --------| ---------|
|BERT Base | 12 |
| BERT Large | 24 |
| MiniLM-L12 | 12 |

<br/>
Although it has 12 layers, it is MUCH smaller than a traditional BERT because it has:

- smaller hidden size;
- compact design;
- fewer parameters.

### **5. V2**

This means that this is the second version of this implementation.


# **5. Scenario I: Semantic Search**

**Semantic Search** is an information retrieval technique that searches for documents **based on their meaning rather than on exact keyword matches**. To achieve this, **queries and documents are represented as vectors in a shared semantic space**, allowing the most relevant items to be retrieved by comparing the similarity between these vectors, even when there is no exact overlap between the words used.


## **5.1. Building a small Knowledge Base**

Now, let's create a small **knowledge base** focusing in the automotive troubleshooting. <br/>
Each document contains, in Portuguese:

| Campo       | Significado       |
| ----------- | ----------------- |
| `id`        | identifier     |
| `categoria` | type of problem  |
| `texto`     | technical description |

<br>

In [8]:
# ============================================================
# Automotive documentation base
# ============================================================

docs = [
    {
        "id": 1,
        "categoria": "bateria",
        "texto": "Quando o carro não dá partida e faz apenas um clique, uma causa comum é bateria descarregada ou mau contato nos terminais."
    },
    {
        "id": 2,
        "categoria": "motor de partida",
        "texto": "Falha no motor de partida pode causar clique ao girar a chave, mesmo quando a bateria está aparentemente carregada."
    },
    {
        "id": 3,
        "categoria": "ignição",
        "texto": "Motor falhando, engasgando ou perdendo força ao acelerar pode estar associado a velas desgastadas, bobinas defeituosas ou falha de ignição."
    },
    {
        "id": 4,
        "categoria": "combustível",
        "texto": "Perda de potência e falhas durante aceleração podem ocorrer por filtro de combustível obstruído ou baixa pressão na linha de combustível."
    },
    {
        "id": 5,
        "categoria": "arrefecimento",
        "texto": "Superaquecimento no trânsito pode estar relacionado a falha da ventoinha, baixo nível de fluido de arrefecimento ou radiador obstruído."
    },
    {
        "id": 6,
        "categoria": "suspensão",
        "texto": "Barulho metálico ao passar em buracos pode indicar desgaste em buchas, bieletas, pivôs ou amortecedores da suspensão."
    },
    {
        "id": 7,
        "categoria": "freios",
        "texto": "Chiado ao frear pode ser causado por pastilhas gastas, disco de freio irregular ou acúmulo de sujeira no sistema de freio."
    },
    {
        "id": 8,
        "categoria": "pneus",
        "texto": "Vibração no volante em velocidades mais altas pode indicar pneus desbalanceados, rodas desalinhadas ou problemas de geometria."
    },
    {
        "id": 9,
        "categoria": "consumo",
        "texto": "Aumento repentino no consumo de combustível pode estar associado a sonda lambda defeituosa, filtro de ar sujo ou velas desgastadas."
    },
    {
        "id": 10,
        "categoria": "elétrica",
        "texto": "Falhas elétricas intermitentes podem ocorrer por mau aterramento, fusíveis oxidados, conectores soltos ou alternador com problema."
    },
]

df = pd.DataFrame(docs)
df

,id,categoria,texto
0,1,bateria,Quando o carro não dá partida e faz apenas um ...
1,2,motor de partida,Falha no motor de partida pode causar clique a...
2,3,ignição,"Motor falhando, engasgando ou perdendo força a..."
3,4,combustível,Perda de potência e falhas durante aceleração ...
4,5,arrefecimento,Superaquecimento no trânsito pode estar relaci...
5,6,suspensão,Barulho metálico ao passar em buracos pode ind...
6,7,freios,Chiado ao frear pode ser causado por pastilhas...
7,8,pneus,Vibração no volante em velocidades mais altas ...
8,9,consumo,Aumento repentino no consumo de combustível po...
9,10,elétrica,Falhas elétricas intermitentes podem ocorrer p...


## **5.2. Generating Embeddings**

Now lets generate the **embeddings** for our small knowledge base! <br>


In [9]:
# ============================================================
# Generate document embeddings
# ============================================================

document_texts = df["texto"].tolist()

doc_embeddings = model.encode(
    document_texts,
    convert_to_tensor=False,
    normalize_embeddings=True
)

print(doc_embeddings.shape)

df2 = pd.DataFrame(doc_embeddings)

df2.style.background_gradient(cmap="Blues")

(10, 384)


The embeddings of our small knowledge base are stored in the `doc_embeddings` matrix!



## **5.3. Semantic Search**

Now, let's implement the **semantic search**:
1. Initially, the **user's questio**n is converted into a **vector embedding** using the **Sentence-BERT**.
2. Then, **cosine similarity** is calculated between the question embedding and the embeddings of the documents in the database.
3. Documents with **greater similarity** are considered semantically closer to the query and are returned as more relevant results.

![Imagem](https://drive.google.com/uc?export=view&id=1Cj8cQ6XyNctFZaokL5tAt18CklmXaW_2)

<br/>


Let's create a **semantic search** function... <br>
This function will use *cosine similarity* to compare a `query_embedding` with the `doc_embedding` created previously.

In [10]:
# ============================================================
# Semantic search function
# ============================================================

def semantic_search(pergunta):
    query_embedding = model.encode(
        [pergunta],
        convert_to_tensor=False,
        normalize_embeddings=True
    )

    similarities = cosine_similarity(
        query_embedding,
        doc_embeddings
    )[0]

    results = df.copy()

    results["similaridade"] = similarities

    results = results.sort_values(
        by="similaridade",
        ascending=False
    )

    return (
        results[["id", "categoria", "similaridade", "texto"]]
        .style
        .hide(axis="index")
        .background_gradient(
            subset=["similaridade"],
            cmap="Blues",
            vmin=0,
            vmax=1
        )
        .format({
            "similaridade": "{:.4f}"
        })
        .set_properties(**{
          "text-align": "left"
    })
    )

## **5.4. Constructing Queries**

Now, let's construct **natural language queries** to validate semantic search! <br>
The goal is to demonstrate that the system can **retrieve documents semantically related** to the user's query, **even when the words used are different** from those present in the database documents.

In [11]:
# ============================================================
# Semantic search
# ============================================================

myquestion = "Meu carro está engasgando quando acelero."
semantic_search(myquestion)


id,categoria,similaridade,texto
1,bateria,0.4752,"Quando o carro não dá partida e faz apenas um clique, uma causa comum é bateria descarregada ou mau contato nos terminais."
4,combustível,0.4453,Perda de potência e falhas durante aceleração podem ocorrer por filtro de combustível obstruído ou baixa pressão na linha de combustível.
8,pneus,0.4414,"Vibração no volante em velocidades mais altas pode indicar pneus desbalanceados, rodas desalinhadas ou problemas de geometria."
7,freios,0.4397,"Chiado ao frear pode ser causado por pastilhas gastas, disco de freio irregular ou acúmulo de sujeira no sistema de freio."
5,arrefecimento,0.4152,"Superaquecimento no trânsito pode estar relacionado a falha da ventoinha, baixo nível de fluido de arrefecimento ou radiador obstruído."
3,ignição,0.3958,"Motor falhando, engasgando ou perdendo força ao acelerar pode estar associado a velas desgastadas, bobinas defeituosas ou falha de ignição."
2,motor de partida,0.3429,"Falha no motor de partida pode causar clique ao girar a chave, mesmo quando a bateria está aparentemente carregada."
9,consumo,0.3387,"Aumento repentino no consumo de combustível pode estar associado a sonda lambda defeituosa, filtro de ar sujo ou velas desgastadas."
10,elétrica,0.2992,"Falhas elétricas intermitentes podem ocorrer por mau aterramento, fusíveis oxidados, conectores soltos ou alternador com problema."
6,suspensão,0.2720,"Barulho metálico ao passar em buracos pode indicar desgaste em buchas, bieletas, pivôs ou amortecedores da suspensão."


In [12]:
# ============================================================
# Semantic search
# ============================================================

myquestion = "O carro só faz um clique e não liga."
semantic_search(myquestion)


id,categoria,similaridade,texto
1,bateria,0.6266,"Quando o carro não dá partida e faz apenas um clique, uma causa comum é bateria descarregada ou mau contato nos terminais."
2,motor de partida,0.5330,"Falha no motor de partida pode causar clique ao girar a chave, mesmo quando a bateria está aparentemente carregada."
4,combustível,0.3363,Perda de potência e falhas durante aceleração podem ocorrer por filtro de combustível obstruído ou baixa pressão na linha de combustível.
8,pneus,0.3209,"Vibração no volante em velocidades mais altas pode indicar pneus desbalanceados, rodas desalinhadas ou problemas de geometria."
3,ignição,0.3042,"Motor falhando, engasgando ou perdendo força ao acelerar pode estar associado a velas desgastadas, bobinas defeituosas ou falha de ignição."
7,freios,0.2774,"Chiado ao frear pode ser causado por pastilhas gastas, disco de freio irregular ou acúmulo de sujeira no sistema de freio."
10,elétrica,0.1858,"Falhas elétricas intermitentes podem ocorrer por mau aterramento, fusíveis oxidados, conectores soltos ou alternador com problema."
5,arrefecimento,0.1752,"Superaquecimento no trânsito pode estar relacionado a falha da ventoinha, baixo nível de fluido de arrefecimento ou radiador obstruído."
6,suspensão,0.1470,"Barulho metálico ao passar em buracos pode indicar desgaste em buchas, bieletas, pivôs ou amortecedores da suspensão."
9,consumo,0.1260,"Aumento repentino no consumo de combustível pode estar associado a sonda lambda defeituosa, filtro de ar sujo ou velas desgastadas."


In [13]:
# ============================================================
# Semantic search
# ============================================================

myquestion = "Quando passo em buracos aparece um barulho metálico."
semantic_search(myquestion)


id,categoria,similaridade,texto
6,suspensão,0.7023,"Barulho metálico ao passar em buracos pode indicar desgaste em buchas, bieletas, pivôs ou amortecedores da suspensão."
10,elétrica,0.4624,"Falhas elétricas intermitentes podem ocorrer por mau aterramento, fusíveis oxidados, conectores soltos ou alternador com problema."
9,consumo,0.3484,"Aumento repentino no consumo de combustível pode estar associado a sonda lambda defeituosa, filtro de ar sujo ou velas desgastadas."
7,freios,0.3052,"Chiado ao frear pode ser causado por pastilhas gastas, disco de freio irregular ou acúmulo de sujeira no sistema de freio."
4,combustível,0.2949,Perda de potência e falhas durante aceleração podem ocorrer por filtro de combustível obstruído ou baixa pressão na linha de combustível.
3,ignição,0.2875,"Motor falhando, engasgando ou perdendo força ao acelerar pode estar associado a velas desgastadas, bobinas defeituosas ou falha de ignição."
5,arrefecimento,0.2827,"Superaquecimento no trânsito pode estar relacionado a falha da ventoinha, baixo nível de fluido de arrefecimento ou radiador obstruído."
2,motor de partida,0.2659,"Falha no motor de partida pode causar clique ao girar a chave, mesmo quando a bateria está aparentemente carregada."
8,pneus,0.2445,"Vibração no volante em velocidades mais altas pode indicar pneus desbalanceados, rodas desalinhadas ou problemas de geometria."
1,bateria,0.2006,"Quando o carro não dá partida e faz apenas um clique, uma causa comum é bateria descarregada ou mau contato nos terminais."


In [14]:
# ============================================================
# Semantic search
# ============================================================

myquestion = "O carro está esquentando muito quando fico parado no trânsito."
semantic_search(myquestion)


id,categoria,similaridade,texto
8,pneus,0.4412,"Vibração no volante em velocidades mais altas pode indicar pneus desbalanceados, rodas desalinhadas ou problemas de geometria."
1,bateria,0.4405,"Quando o carro não dá partida e faz apenas um clique, uma causa comum é bateria descarregada ou mau contato nos terminais."
5,arrefecimento,0.3970,"Superaquecimento no trânsito pode estar relacionado a falha da ventoinha, baixo nível de fluido de arrefecimento ou radiador obstruído."
7,freios,0.3654,"Chiado ao frear pode ser causado por pastilhas gastas, disco de freio irregular ou acúmulo de sujeira no sistema de freio."
4,combustível,0.2791,Perda de potência e falhas durante aceleração podem ocorrer por filtro de combustível obstruído ou baixa pressão na linha de combustível.
2,motor de partida,0.2553,"Falha no motor de partida pode causar clique ao girar a chave, mesmo quando a bateria está aparentemente carregada."
9,consumo,0.2435,"Aumento repentino no consumo de combustível pode estar associado a sonda lambda defeituosa, filtro de ar sujo ou velas desgastadas."
6,suspensão,0.2196,"Barulho metálico ao passar em buracos pode indicar desgaste em buchas, bieletas, pivôs ou amortecedores da suspensão."
3,ignição,0.2186,"Motor falhando, engasgando ou perdendo força ao acelerar pode estar associado a velas desgastadas, bobinas defeituosas ou falha de ignição."
10,elétrica,0.1798,"Falhas elétricas intermitentes podem ocorrer por mau aterramento, fusíveis oxidados, conectores soltos ou alternador com problema."


## **5.5. Keyword-based search**

Now, we will implement a **keyword-based search** to serve as a comparison with the **semantic search**. <br>
In this method, the system only checks the **literal occurrence of the query words in the documents**, without understanding the meaning of the text. <br>
The objective is to **show the limitations of the traditional** approach compared to semantic embeddings.

In [15]:
# ============================================================
# Simple keyword search function for comparison
# ============================================================

import pandas as pd

def keyword_search(query):
    terms = query.lower().split()
    results = []

    for _, row in df.iterrows():
        text = row["texto"].lower()
        matched_terms = [
            term
            for term in terms
            if term in text
        ]
        score = len(matched_terms)
        results.append({
            "category": row["categoria"],
            "keyword_score": score,
            "matched_terms": ", ".join(matched_terms),
            "text": row["texto"]
        })
    results = pd.DataFrame(results)
    results = results.sort_values(
        by="keyword_score",
        ascending=False
    )
    results = results.reset_index(drop=True)
    results.insert(
        0,
        "rank",
        range(1, len(results) + 1)
    )
    return (
        results[
            [
                "rank",
                "category",
                "keyword_score",
                "matched_terms",
                "text"
            ]
        ]
        .style
        .hide(axis="index")
        .background_gradient(
            subset=["keyword_score"],
            cmap="Reds",
            vmin=0,
            vmax=len(terms)
        )
        .format({
            "keyword_score": "{:.0f}"
        })
        .set_properties(**{
            "text-align": "left"
        })
        .set_table_styles([
            {
                "selector": "th",
                "props": [
                    ("text-align", "left")
                ]
            }
        ])
    )

## **5.6. Didactic comparison**

Now, let's demonstrate the difference between **traditional keyword search** and **semantic search based on embeddings**, highlighting how Transformers can compare meanings instead of just literal words.

In [16]:
# ============================================================
# 9. Didactic comparison
# ============================================================

myquestion = "Meu carro está engasgando quando acelero."

print("Keyword Search:")
display(keyword_search(myquestion))

print("\nSemantic Search:")
display(semantic_search(myquestion))

Keyword Search:


rank,category,keyword_score,matched_terms,text
1,bateria,2,"carro, quando","Quando o carro não dá partida e faz apenas um clique, uma causa comum é bateria descarregada ou mau contato nos terminais."
2,motor de partida,2,"está, quando","Falha no motor de partida pode causar clique ao girar a chave, mesmo quando a bateria está aparentemente carregada."
3,ignição,1,engasgando,"Motor falhando, engasgando ou perdendo força ao acelerar pode estar associado a velas desgastadas, bobinas defeituosas ou falha de ignição."
4,combustível,0,,Perda de potência e falhas durante aceleração podem ocorrer por filtro de combustível obstruído ou baixa pressão na linha de combustível.
5,arrefecimento,0,,"Superaquecimento no trânsito pode estar relacionado a falha da ventoinha, baixo nível de fluido de arrefecimento ou radiador obstruído."
6,suspensão,0,,"Barulho metálico ao passar em buracos pode indicar desgaste em buchas, bieletas, pivôs ou amortecedores da suspensão."
7,freios,0,,"Chiado ao frear pode ser causado por pastilhas gastas, disco de freio irregular ou acúmulo de sujeira no sistema de freio."
8,pneus,0,,"Vibração no volante em velocidades mais altas pode indicar pneus desbalanceados, rodas desalinhadas ou problemas de geometria."
9,consumo,0,,"Aumento repentino no consumo de combustível pode estar associado a sonda lambda defeituosa, filtro de ar sujo ou velas desgastadas."
10,elétrica,0,,"Falhas elétricas intermitentes podem ocorrer por mau aterramento, fusíveis oxidados, conectores soltos ou alternador com problema."



Semantic Search:


id,categoria,similaridade,texto
1,bateria,0.4752,"Quando o carro não dá partida e faz apenas um clique, uma causa comum é bateria descarregada ou mau contato nos terminais."
4,combustível,0.4453,Perda de potência e falhas durante aceleração podem ocorrer por filtro de combustível obstruído ou baixa pressão na linha de combustível.
8,pneus,0.4414,"Vibração no volante em velocidades mais altas pode indicar pneus desbalanceados, rodas desalinhadas ou problemas de geometria."
7,freios,0.4397,"Chiado ao frear pode ser causado por pastilhas gastas, disco de freio irregular ou acúmulo de sujeira no sistema de freio."
5,arrefecimento,0.4152,"Superaquecimento no trânsito pode estar relacionado a falha da ventoinha, baixo nível de fluido de arrefecimento ou radiador obstruído."
3,ignição,0.3958,"Motor falhando, engasgando ou perdendo força ao acelerar pode estar associado a velas desgastadas, bobinas defeituosas ou falha de ignição."
2,motor de partida,0.3429,"Falha no motor de partida pode causar clique ao girar a chave, mesmo quando a bateria está aparentemente carregada."
9,consumo,0.3387,"Aumento repentino no consumo de combustível pode estar associado a sonda lambda defeituosa, filtro de ar sujo ou velas desgastadas."
10,elétrica,0.2992,"Falhas elétricas intermitentes podem ocorrer por mau aterramento, fusíveis oxidados, conectores soltos ou alternador com problema."
6,suspensão,0.2720,"Barulho metálico ao passar em buracos pode indicar desgaste em buchas, bieletas, pivôs ou amortecedores da suspensão."


## **5.7. Building a simple interface**

Now, let's implement a simple interface to process our questions formulated in **Natural Language**.

In [17]:
# ============================================================
# 10. Interface simples para testar perguntas
# ============================================================

from IPython.display import display

myquestion = input(
    "Describe the car problem or type 'exit': "
)

print("\nSemantic Search:\n")
display(
    semantic_search(myquestion)
)

print("\nKeyword Search:\n")
display(
    keyword_search(myquestion)
)

Describe the car problem or type 'exit': Fault

Semantic Search:



id,categoria,similaridade,texto
10,elétrica,0.3772,"Falhas elétricas intermitentes podem ocorrer por mau aterramento, fusíveis oxidados, conectores soltos ou alternador com problema."
3,ignição,0.3435,"Motor falhando, engasgando ou perdendo força ao acelerar pode estar associado a velas desgastadas, bobinas defeituosas ou falha de ignição."
7,freios,0.3129,"Chiado ao frear pode ser causado por pastilhas gastas, disco de freio irregular ou acúmulo de sujeira no sistema de freio."
6,suspensão,0.2942,"Barulho metálico ao passar em buracos pode indicar desgaste em buchas, bieletas, pivôs ou amortecedores da suspensão."
4,combustível,0.2765,Perda de potência e falhas durante aceleração podem ocorrer por filtro de combustível obstruído ou baixa pressão na linha de combustível.
5,arrefecimento,0.2389,"Superaquecimento no trânsito pode estar relacionado a falha da ventoinha, baixo nível de fluido de arrefecimento ou radiador obstruído."
1,bateria,0.2305,"Quando o carro não dá partida e faz apenas um clique, uma causa comum é bateria descarregada ou mau contato nos terminais."
8,pneus,0.2291,"Vibração no volante em velocidades mais altas pode indicar pneus desbalanceados, rodas desalinhadas ou problemas de geometria."
9,consumo,0.1812,"Aumento repentino no consumo de combustível pode estar associado a sonda lambda defeituosa, filtro de ar sujo ou velas desgastadas."
2,motor de partida,0.1650,"Falha no motor de partida pode causar clique ao girar a chave, mesmo quando a bateria está aparentemente carregada."



Keyword Search:



rank,category,keyword_score,matched_terms,text
1,bateria,0,,"Quando o carro não dá partida e faz apenas um clique, uma causa comum é bateria descarregada ou mau contato nos terminais."
2,motor de partida,0,,"Falha no motor de partida pode causar clique ao girar a chave, mesmo quando a bateria está aparentemente carregada."
3,ignição,0,,"Motor falhando, engasgando ou perdendo força ao acelerar pode estar associado a velas desgastadas, bobinas defeituosas ou falha de ignição."
4,combustível,0,,Perda de potência e falhas durante aceleração podem ocorrer por filtro de combustível obstruído ou baixa pressão na linha de combustível.
5,arrefecimento,0,,"Superaquecimento no trânsito pode estar relacionado a falha da ventoinha, baixo nível de fluido de arrefecimento ou radiador obstruído."
6,suspensão,0,,"Barulho metálico ao passar em buracos pode indicar desgaste em buchas, bieletas, pivôs ou amortecedores da suspensão."
7,freios,0,,"Chiado ao frear pode ser causado por pastilhas gastas, disco de freio irregular ou acúmulo de sujeira no sistema de freio."
8,pneus,0,,"Vibração no volante em velocidades mais altas pode indicar pneus desbalanceados, rodas desalinhadas ou problemas de geometria."
9,consumo,0,,"Aumento repentino no consumo de combustível pode estar associado a sonda lambda defeituosa, filtro de ar sujo ou velas desgastadas."
10,elétrica,0,,"Falhas elétricas intermitentes podem ocorrer por mau aterramento, fusíveis oxidados, conectores soltos ou alternador com problema."


___

# **6. Scenario II: Retrieving multiple chunks**

Now let's implement a **Encoder-Only Transformer** with **chunks retrieving** for the Automotive Troubleshooting System based on BERT.


## **6.1. Installing libraries**

First of all, let's install some libraries.

In [18]:
# ============================================================
# Installing libraries
# ============================================================

!pip install -q pypdf sentence-transformers transformers

import requests   # Download of external resources (e.g., manuals, datasets, PDFs)
import textwrap   # Formatting and wrapping long text passages for visualization
import torch      # Tensor manipulation and hardware acceleration (CPU/GPU)

from pypdf import PdfReader         # PDF reading and text extraction (Knowledge Base creation)
from sentence_transformers import SentenceTransformer, util   # Sentence embeddings generation and similarity computation
from transformers import pipeline   # Transformer pipelines for Question Answering


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 7.3 MB/s eta 0:00:00


## **6.2. Loading/Reusing BERT**

Now let's load (or reuse) our BERT...

In [19]:
# ------------------------------------------------------------
# Load or reuse BERT sentence embedding model
# ------------------------------------------------------------

try:
    model
    print("Using previously loaded SentenceTransformer model.")
except NameError:
    model = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    print("Loaded new SentenceTransformer model.")


Using previously loaded SentenceTransformer model.


## **6.3. Selecting the Knowledge Base**

Instead of using a simple database with few entries, we will now use a more complex database **automatically extracted from documents stored in an external folder**. <br>
To demonstrate our system, we will use a collection of manuals as our new **corpus**.

In [20]:
# ------------------------------------------------------------
# RAG - building a Knowledge Base from an external folder
# ------------------------------------------------------------

!pip install -q gdown

import os
import io
import gdown

from pypdf import PdfReader
from tqdm.notebook import tqdm

folder_url = "https://drive.google.com/drive/folders/1dNVwyidMhZ0NHm7qqXDVgneInaoSuL4b?usp=sharing"

download_dir = "automotive_manuals"

# ------------------------------------------------------------
# Download folder contents locally
# ------------------------------------------------------------

gdown.download_folder(
    folder_url,
    output=download_dir,
    quiet=False,
    use_cookies=False
)

# ------------------------------------------------------------
# Find all PDF files
# ------------------------------------------------------------

pdf_files = []

for root, dirs, files in os.walk(download_dir):
    for file in files:
        if file.lower().endswith(".pdf"):
            pdf_files.append(
                os.path.join(root, file)
            )

pdf_files = sorted(pdf_files)

print(f"{len(pdf_files)} PDF files found:\n")

for pdf_path in pdf_files:
    print("-", os.path.basename(pdf_path))

# ------------------------------------------------------------
# Build Knowledge Base
# ------------------------------------------------------------

knowledge_base = []

total_files = len(pdf_files)

for file_number, pdf_path in enumerate(pdf_files, start=1):

    pdf_name = os.path.basename(pdf_path)

    print(
        f"\nProcessing file {file_number} of {total_files}: "
        f"{pdf_name}"
    )

    try:

        with open(pdf_path, "rb") as f:
            pdf_stream = io.BytesIO(
                f.read()
            )

        reader = PdfReader(pdf_stream)

        total_pages = len(reader.pages)

        for page_number, page in enumerate(
            tqdm(
                reader.pages,
                desc=pdf_name,
                leave=False
            ),
            start=1
        ):

            text = page.extract_text()

            if text:

                knowledge_base.append({
                    "source_file": pdf_name,
                    "page": page_number,
                    "text": text
                })

        print(
            f"  -> {total_pages} pages processed"
        )

    except Exception as e:

        print(
            f"Error processing {pdf_name}: {e}"
        )

print(
    f"\nLoaded {len(knowledge_base)} pages "
    f"from {total_files} PDF files."
)

ERROR: Operation cancelled by user


Retrieving folder contents


Processing file 1naVXpQhnQBFN-5KggbK02bJw1lacjTsD Apostila_SENAI_motores_de_combustao_interna.pdf
Processing file 1s5r1DWUK3_ve5xaMW6a_0IJpx7GUwf2j Manual_Chevrolet_Blazer_2024.pdf
Processing file 1zngwxi7g2IO7jc1w2n0HQa-iyYUAJYy_ Manual_Chevrolet_Captiva_2026.pdf
Processing file 1VtrunkNf20d4nGZza-Gch17TYg7EQSIO Manual_Chevrolet_Equinox_2024.pdf
Processing file 18mHVloxBChBOwQh8sFl_fpHIB9iA4Cd1 Manual_Chevrolett_Spark_2026.pdf
Processing file 1d_hUlrnJWDmcskEbqvH5Liu5k0AfCoq1 Manual_Honda_City.pdf
Processing file 1fH2Y9EQnGOkwweEKGA_IokbpWoaCE7sj Manual_SENAI_Técnico_manutencao_automotiva.pdf
Processing file 1QYj6RaknZcDuEx1FFGIvSPRvrcxTiEP6 Manual_Toyota_SW4_2026.pdf
Processing file 1sI1hSk93LLg-95S-s9Ot8klfNHxzQLZv Manual_VW_Amarok_2026.pdf
Processing file 1Y5v7sK97OQpr2wpZ5pFkIsJ0rq8s0HzD Manual_VW_Fox_2022.pdf
Processing file 1sP6M-f1E_Fdei47gb4r0joBHZIYq4dNF Manual_VW_Gol_2023.pdf
Processing file 1liEQwNHiNahU2xpzHMHQl5GIORYrC_Ec Manual_VW_Polo_2026.pdf
Processing file 14Qxh0BWir

Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1naVXpQhnQBFN-5KggbK02bJw1lacjTsD
To: /content/automotive_manuals/Apostila_SENAI_motores_de_combustao_interna.pdf
100%|██████████| 6.06M/6.06M [00:00<00:00, 14.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1s5r1DWUK3_ve5xaMW6a_0IJpx7GUwf2j
To: /content/automotive_manuals/Manual_Chevrolet_Blazer_2024.pdf
100%|██████████| 4.46M/4.46M [00:00<00:00, 24.9MB/s]
Downloading...
From: https://drive.google.com/uc?id=1zngwxi7g2IO7jc1w2n0HQa-iyYUAJYy_
To: /content/automotive_manuals/Manual_Chevrolet_Captiva_2026.pdf
100%|██████████| 23.2M/23.2M [00:00<00:00, 25.8MB/s]
Downloading...
From: https://drive.google.com/uc?id=1VtrunkNf20d4nGZza-Gch17TYg7EQSIO
To: /content/automotive_manuals/Manual_Chevrolet_Equinox_2024.pdf
100%|██████████| 5.00M/5.00M [00:00<00:00, 28.8MB/s]
Downloading...
From: https://drive.google.com/uc?id=18mHVloxBC

18 PDF files found:

- Apostila_SENAI_motores_de_combustao_interna.pdf
- Manual-VW-Tiguan-2026.pdf
- Manual_Chevrolet_Blazer_2024.pdf
- Manual_Chevrolet_Captiva_2026.pdf
- Manual_Chevrolet_Equinox_2024.pdf
- Manual_Chevrolett_Spark_2026.pdf
- Manual_Honda_City.pdf
- Manual_SENAI_Técnico_manutencao_automotiva.pdf
- Manual_Toyota_SW4_2026.pdf
- Manual_VW_Amarok_2026.pdf
- Manual_VW_Fox_2022.pdf
- Manual_VW_Gol_2023.pdf
- Manual_VW_Polo_2026.pdf
- Manual_VW_Saveiro-2027.pdf
- Manual_VW_T-Cross_2026.pdf
- Manual_VW_Taos_2026.pdf
- Manual_VW_Tera_2026.pdf
- Manual_VW_Voyage_2023.pdf

Processing file 1 of 18: Apostila_SENAI_motores_de_combustao_interna.pdf


...

  -> 356 pages processed

Processing file 17 of 18: Manual_VW_Tera_2026.pdf


Manual_VW_Tera_2026.pdf:   0%|          | 0/270 [00:00<?, ?it/s]

  -> 270 pages processed

Processing file 18 of 18: Manual_VW_Voyage_2023.pdf


Manual_VW_Voyage_2023.pdf:   0%|          | 0/295 [00:00<?, ?it/s]

  -> 295 pages processed

Loaded 5691 pages from 18 PDF files.


Let's take a closer look at the files processed...

In [21]:
# --------------------------------------
# RAG - Number of manuals processed
# --------------------------------------

import pandas as pd

df_knowledge_base = pd.DataFrame(
    knowledge_base
)

summary = (
    df_knowledge_base
    .groupby("source_file")
    .size()
    .reset_index(name="pages")
    .sort_values("pages", ascending=False)
)

display(summary)

print(
    "\nTotal pages processed:",
    summary["pages"].sum()
)


,source_file,pages
8,Manual_Toyota_SW4_2026.pdf,656
6,Manual_Honda_City.pdf,614
1,Manual-VW-Tiguan-2026.pdf,382
4,Manual_Chevrolet_Equinox_2024.pdf,365
15,Manual_VW_Taos_2026.pdf,356
2,Manual_Chevrolet_Blazer_2024.pdf,356
10,Manual_VW_Fox_2022.pdf,297
17,Manual_VW_Voyage_2023.pdf,295
11,Manual_VW_Gol_2023.pdf,293
9,Manual_VW_Amarok_2026.pdf,285



Total pages processed: 5691


## **6.4. Chunks generation**

Due to practical limitations, the BERT model used in this lesson can process at most 512 tokens at a time. Since automotive manuals are much larger than this limit, the documents must be divided into **smaller text segments** called **chunks** before embeddings can be generated. **Each chunk will generate one embedding** in next phases.

To preserve context across chunk boundaries, consecutive chunks **share a small portion of text through an overlap mechanism**. This reduces the risk of losing important information that might otherwise be split between two adjacent chunks.

In [22]:
# ------------------------------------------------------------
# Create text chunks per source file, preserving overlap
# across page boundaries and keeping page references
# ------------------------------------------------------------

import pandas as pd

CHUNK_SIZE = 800
OVERLAP = 150

def create_chunks(
    knowledge_base,
    chunk_size=CHUNK_SIZE,
    overlap=OVERLAP
):

    chunks = []

    df_kb = pd.DataFrame(knowledge_base)

    for source_file, group in df_kb.groupby("source_file"):

        group = group.sort_values("page")

        all_words = []

        for _, row in group.iterrows():

            page_number = row["page"]
            words = row["text"].split()

            for word in words:
                all_words.append({
                    "word": word,
                    "page": page_number
                })

        start = 0
        chunk_id = 0

        while start < len(all_words):

            end = start + chunk_size

            chunk_words = all_words[start:end]

            chunk_text = " ".join(
                item["word"]
                for item in chunk_words
            )

            pages_in_chunk = [
                item["page"]
                for item in chunk_words
            ]

            if len(chunk_text.strip()) > 0:

                chunks.append({
                    "source_file": source_file,
                    "chunk_id": chunk_id,
                    "start_page": min(pages_in_chunk),
                    "end_page": max(pages_in_chunk),
                    "start_word": start,
                    "end_word": min(end, len(all_words)),
                    "text": chunk_text
                })

            start += chunk_size - overlap
            chunk_id += 1

    return chunks


chunks = create_chunks(
    knowledge_base,
    chunk_size=CHUNK_SIZE,
    overlap=OVERLAP
)

print(f"Created {len(chunks)} chunks.")


Created 3327 chunks.


Now let's create a function `show_chunk` that allow us to see the contents of the chunks...

In [23]:
# -----------------------------------
# Display chunk contents
# -----------------------------------

import textwrap

def show_chunk(chunk_index):

    if chunk_index < 0 or chunk_index >= len(chunks):
        print(
            f"Invalid chunk index. "
            f"Valid range: 0 to {len(chunks)-1}"
        )
        return

    chunk = chunks[chunk_index]

    print("=" * 100)
    print(f"Chunk Index : {chunk_index}")
    print(f"Source File : {chunk['source_file']}")

    if chunk["start_page"] == chunk["end_page"]:
        print(f"Page        : {chunk['start_page']}")
    else:
        print(
            f"Pages       : "
            f"{chunk['start_page']} - {chunk['end_page']}"
        )

    print(f"Words       : {len(chunk['text'].split())}")
    print("=" * 100)

    print(
        textwrap.fill(
            chunk["text"],
            width=100
        )
    )

    print("=" * 100)
    print()


Let's check some of our chunks...

In [24]:
show_chunk(10)
show_chunk(11)

Chunk Index : 10
Source File : Apostila_SENAI_motores_de_combustao_interna.pdf
Pages       : 44 - 49
Words       : 800
a razão entre o volume do cilindro situado acima do PMI e aquele que fica acima do PMS. Rc = V + v v
Figura 69 – Relação de compressão Fonte: Transparências de Motores FIAT A relação de compressão (RC)
indica quantas vezes a mistura é comprimida quando o êmbolo passa do PMI ao PMS. Quanto maior a RC,
maior a potência do motor. Os motores a álcool possuem uma relação de compressão maior que os a
gasolina, devido às características do combustível. A unidade de medida é uma relação entre volumes
e é dada por um número. Exemplo: 8:1 Onde: V = cilindrada π = 3,14 r = raio do cilindro em cm h =
curso do êmbolo n = número de cilindros do motor Onde: Rc = relação de compressão V = cilindrada v =
volume da câmara de compressão 45 3.7.3- Torque A palavra torque quer dizer torção. O torque depende
não só da força que é aplicada, como da distância que funciona como braço de alavan

## **6.5. Embedding generation**

Now, let's generate the **embeddings** for your **knowledge base**!


In [25]:
# ------------------------------------------------------------
# Generate or reuse embeddings for manual chunks
# ------------------------------------------------------------

texts = [
    chunk["text"]
    for chunk in chunks
]

# Reuse embeddings only if they match the current chunk list
try:
    embeddings_are_valid = (
        chunk_embeddings.shape[0] == len(texts)
    )

    if embeddings_are_valid:
        print("Using previously generated chunk embeddings.")
    else:
        print("Chunk list changed. Regenerating embeddings...")

        chunk_embeddings = model.encode(
            texts,
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=True
        )

        base_embeddings = chunk_embeddings.clone()

        print("Embeddings regenerated.")
        print("Base embeddings updated.")

except NameError:
    chunk_embeddings = model.encode(
        texts,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    base_embeddings = chunk_embeddings.clone()

    print("Embeddings generated.")
    print("Base embeddings stored.")

# Ensure base_embeddings exists and matches the current embeddings
try:
    if base_embeddings.shape[0] != chunk_embeddings.shape[0]:
        base_embeddings = chunk_embeddings.clone()
        print("Base embeddings updated.")
except NameError:
    base_embeddings = chunk_embeddings.clone()
    print("Base embeddings stored.")

print("Embedding matrix shape:", chunk_embeddings.shape)


Batches:   0%|          | 0/104 [00:00<?, ?it/s]

Embeddings generated.
Base embeddings stored.
Embedding matrix shape: torch.Size([3327, 384])


## **6.6. Semantic Retrieval**

Now let's create a **function** able to process a **semantic retrieval** with our chunks.

The embedding generated from a **user's query** is compared against the embeddings of all **document chunks**. The chunks with the highest semantic similarity scores are retrieved as the most relevant pieces of information.

In [26]:
# ------------------------------------------------------------
# Semantic retrieval with BERT embeddings
# ------------------------------------------------------------

NUM_RETRIEVED = 5

def retrieve_relevant_chunks(query, top_k=NUM_RETRIEVED):

    query_embedding = model.encode(
        query,
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    top_k = min(top_k, len(chunks))

    similarities = util.cos_sim(
        query_embedding,
        chunk_embeddings
    )[0]

    top_results = similarities.topk(k=top_k)

    retrieved = []

    for score, idx in zip(top_results.values, top_results.indices):

        idx = int(idx)
        chunk = chunks[idx]

        retrieved.append({
            "source_file": chunk["source_file"],
            "chunk_id": chunk["chunk_id"],
            "start_page": chunk["start_page"],
            "end_page": chunk["end_page"],
            "score": float(score),
            "text": chunk["text"]
        })

    return retrieved


Here we will create a function to organize the presentation of the BERT results printing a header and the retrieved texts...

In [27]:
# ------------------------------------------------------------
# Encoder-only troubleshooting report
# ------------------------------------------------------------

import textwrap

def encoder_only_troubleshooting_report(
    query,
    top_k=5,
    preview_chars=1200
):

    retrieved = retrieve_relevant_chunks(
        query,
        top_k=top_k
    )

    print("=" * 100)
    print("AUTOMOTIVE TROUBLESHOOTING ASSISTANT")
    print("Encoder-Only Transformer Retrieval")
    print("=" * 100)

    print("\nUSER SYMPTOM:")
    print(
        textwrap.fill(
            query.strip(),
            width=100
        )
    )

    print("\nMOST RELEVANT MANUAL SECTIONS:")

    if len(retrieved) == 0:
        print(
            "No relevant chunks were retrieved. "
            "The knowledge base may be empty or embeddings may not have been generated."
        )
        return retrieved

    for i, r in enumerate(retrieved, start=1):

        print("-" * 100)

        if r["start_page"] == r["end_page"]:
            page_info = f"Page: {r['start_page']}"
        else:
            page_info = (
                f"Pages: {r['start_page']} - {r['end_page']}"
            )

        print(
            f"{i}. "
            f"Source: {r['source_file']} | "
            f"Chunk: {r['chunk_id']} | "
            f"{page_info} | "
            f"Similarity: {r['score']:.3f}"
        )

        print("\nCHUNK PREVIEW:\n")

        print(
            textwrap.fill(
                r["text"][:preview_chars],
                width=100
            )
        )

    return



In [28]:
# ------------------------------------------------------------
# Semantic Retrieval from the Knowledge Base
# ------------------------------------------------------------

# Example user query
query = "Acendeu a luz do motor no meu painel."

# Display a complete retrieval report
encoder_only_troubleshooting_report(
    query=query,
    top_k=5,
    preview_chars=1200
)

AUTOMOTIVE TROUBLESHOOTING ASSISTANT
Encoder-Only Transformer Retrieval

USER SYMPTOM:
Acendeu a luz do motor no meu painel.

MOST RELEVANT MANUAL SECTIONS:
----------------------------------------------------------------------------------------------------
1. Source: Manual-VW-Tiguan-2026.pdf | Chunk: 92 | Pages: 125 - 126 | Similarity: 0.644

CHUNK PREVIEW:

no momento do desli- gamento automático da ignição, a luz de posição se acende até o travamento do
veículo ou por no máxi- mo cerca de 30 minutos. Função de nova partida do motor Se uma chave do
veículo válida não for reconhecida no interior do veículo após o desligamento involun- tário do
motor, será possível dar partida no motor novamente em aproximadamente cinco segundos. Decorrido
este tempo, não é mais possível ligar o motor sem uma chave do veículo válida no interior do
veículo. ATENÇÃO Se o pedal do freio for acionado ao ligar a ignição, o motor dá partida
imediatamente. Isso pode oca- sionar mo vimentos indesejados do veí

___
# **7. Scenario III: BERT Fine-Tuning**

In the context of BERT-based semantic search, **fine-tuning** consists of **updating the encoder weights** using domain-specific documents so that **semantically related concepts become closer in the embedding space**. As a result, the model learns a more specialized representation of the domain, improving the retrieval of relevant information while still relying on the knowledge base for factual content.

**Fundamental premise** of the training: Adjacent chunks within a document typically discuss the same topic or closely related concepts. Therefore, during fine-tuning, the model is encouraged to produce embeddings for neighboring chunks that are closer in the vector space, reinforcing their semantic similarity. <br>
<br>

<div align="center">
  <img
    src="https://drive.google.com/uc?export=view&id=13paqC55B1LAy60eUxIZe0A-ZXcVJ5rI0"
    width="900">
</div>

<br>

It is importante to note that **BERT continues to be an encoder** (text → embedding), not a decoder (question → answer). That is, the fine-tuned model can generate embeddings that can better answer the question: <br>
<br>
"*What should be checked when there is an electrical fault?*" <br>

However, the model can't answer directly:<br>

 "*check: fuse, ground connector, voltage supply, plug connections*". <br>

In other words:

- Before fine-tuning: <br>
Question → BERT → Embedding → Knowledge base → Retrieved chunks

- After fine-tuning: <br>
Question → Fine-tuned BERT → Better embedding → Knowledge Base → Retrieved chunks

The main advantage of the fine-tuning is **the increase in retrieval quality**, **not the elimination of the knowledge base**!


## **7.1. Intalling libraries**

Let's start installing some libraries...

In [29]:
# ============================================================
# Install
# ============================================================

!pip install -q sentence-transformers pypdf

import requests
import random
import pandas as pd
import textwrap
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, InputExample, losses, util
from torch.utils.data import DataLoader


/tmp/ipykernel_2349/2700162799.py:12: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses, util


## **7.2. Loading/Reusing BERT**

Let's load (or reuse) our BERT...


In [30]:
# ------------------------------------------------------------
# Reuse the already loaded BERT/Sentence-BERT model
# ------------------------------------------------------------

try:
    model
    print("Using previously loaded SentenceTransformer model.")
except NameError:
    model = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    print("Loaded new SentenceTransformer model.")

Using previously loaded SentenceTransformer model.


## **7.3. Reusing chunks**

Let's **reuse** our previouly extracted **chunks**...

In [31]:
# ------------------------------------------------------------
# Reuse previously created chunks
# ------------------------------------------------------------

CHUNK_EXAMPLE = 120

try:
    chunks
    print(f"Using existing chunks: {len(chunks)} chunks.\n")
    print("Example chunk:\n")
    show_chunk(CHUNK_EXAMPLE)
except NameError:
    raise RuntimeError(
        "Chunks were not found. "
        "Please execute the 'Create text chunks' section first."
    )


Using existing chunks: 3327 chunks.

Example chunk:

Chunk Index : 120
Source File : Manual-VW-Tiguan-2026.pdf
Pages       : 122 - 123
Words       : 800
no motor novamente e conduzir um trecho curto devagar. 3. Observar mensagens no display do
instrumento combinado. 4. Caso as luzes de controle continuem acesas, procurar uma empresa
especializada e qualifica- da e mandar verificar o sistema. A Volkswagen recomenda procurar uma
Concessionária Volkswagen. Pedais Informações sobre os pedais Fig. 86 Na área para os pés: pedais. 1
Pedal do acelerador. 2 Pedal do freio. ATENÇÃO Objetos na área para os pés do condutor podem
obstruir o acionamento dos pedais. Isto pode cau- sar perda de controle do v eículo e aumenta o
risco de ferimentos graves ou fatais. · Atentar para que todos os pedais sempre pos- sam ser
acionados sem impedimentos. · Use apenas tapetes adequados para o seu veí- culo. · Fixe sempre os
tapetes de maneira segura na área para os pés. · Nunca colocar tapetes ou outros revesti

## **7.4. Preparing the self-supervised training**

Unlike traditional text classification models, Sentence-BERT **learns by comparing pairs of texts rather than processing individual documents** in isolation (this is a encoder-only Transformer!!). Therefore, the training process requires examples that indicate which texts should produce **similar embeddings**.

In our implementation, neighboring chunks from the same automotive manual are treated as **positive training pairs**, since they usually discuss related topics and share overlapping content. During fine-tuning, the model **learns to move the embeddings of these paired chunks closer together** in the semantic space, resulting in a representation that is better adapted to the structure and terminology of automotive manuals and, consequently, improves semantic retrieval performance.

<img src="https://drive.google.com/uc?export=view&id=1g0B1Ec1GFuimu4WebEgyNKjxBK3bxNWQ" width="900">

In [32]:
# ------------------------------------------------------------
# Create positive training pairs from neighboring chunks
# and keep metadata to visualize the overlap
# ------------------------------------------------------------

from sentence_transformers import InputExample
import random
import textwrap

train_examples = []

# Ensure chunks are ordered by source file and chunk id
sorted_chunks = sorted(
    chunks,
    key=lambda x: (
        x["source_file"],
        x["chunk_id"]
    )
)

for i in range(len(sorted_chunks) - 1):

    current_chunk = sorted_chunks[i]
    next_chunk = sorted_chunks[i + 1]

    # Create positive pairs only within the same source document
    if current_chunk["source_file"] == next_chunk["source_file"]:

        train_examples.append({
            "example": InputExample(
                texts=[
                    current_chunk["text"],
                    next_chunk["text"]
                ]
            ),
            "chunk_a": current_chunk,
            "chunk_b": next_chunk
        })

random.shuffle(train_examples)

print(f"Training pairs created: {len(train_examples)}")


Training pairs created: 3309


Let's see one of these training pairs...

In [33]:
# ------------------------------------------------------------
# Display a training pair and its overlap
# ------------------------------------------------------------

def show_training_pair(pair_index, overlap_words=150):

    if pair_index < 0 or pair_index >= len(train_examples):
        print(
            f"Invalid pair index. "
            f"Valid range: 0 to {len(train_examples)-1}"
        )
        return

    pair = train_examples[pair_index]

    chunk_a = pair["chunk_a"]
    chunk_b = pair["chunk_b"]

    words_a = chunk_a["text"].split()
    words_b = chunk_b["text"].split()

    overlap_a = words_a[-overlap_words:]
    overlap_b = words_b[:overlap_words]

    print("=" * 100)
    print(f"Training Pair : {pair_index}")
    print("=" * 100)

    print(f"Source file : {chunk_a['source_file']}")
    print(
        f"Chunk A     : {chunk_a['chunk_id']} "
        f"| Pages {chunk_a['start_page']} - {chunk_a['end_page']}"
    )
    print(
        f"Chunk B     : {chunk_b['chunk_id']} "
        f"| Pages {chunk_b['start_page']} - {chunk_b['end_page']}"
    )

    print("\n" + "-" * 100)
    print("End of Chunk A")
    print("-" * 100)
    print(
        textwrap.fill(
            " ".join(overlap_a),
            width=100
        )
    )

    print("\n" + "-" * 100)
    print("Beginning of Chunk B")
    print("-" * 100)
    print(
        textwrap.fill(
            " ".join(overlap_b),
            width=100
        )
    )

    print("=" * 100)

show_training_pair(0)

Training Pair : 0
Source file : Manual_VW_Gol_2023.pdf
Chunk A     : 101 | Pages 151 - 153
Chunk B     : 102 | Pages 152 - 156

----------------------------------------------------------------------------------------------------
End of Chunk A
----------------------------------------------------------------------------------------------------
ou Baixo. ATENÇÃO A distração do condutor pode causar acidentes e f erimentos. ● Jamais efetuar
configurações durante a con- dução.  Manual de instruções150 Media Plus (RCD 320G 2 DIN) Rádio Menu
principal RADIO Fig. 125 Menu principal RADIO. Dependendo da versão do veículo, o rádio pode não
estar disponív el. O sistema de rádio é fornecido conforme o país e os equipamentos em diversas
variantes de apare- lhos. Na vista geral do aparelho estão relaciona- das todas as possíveis
variantes de aparelho . O acesso à operação do rádio e ao comando de- pendem, em parte, do aparelho.
Pressionar os botões do aparelho FM ou AM pa- ra iniciar a operação do 

During **fine-tuning**, the model learns to bring the embeddings of these overlapping chunks **closer together** in the semantic space.

Now, lets organize the chunk pairs in small batches for training.

In [34]:
# ------------------------------------------------------------
# DataLoader
# ------------------------------------------------------------

from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    [item["example"] for item in train_examples],
    shuffle=True,
    batch_size=8
)


## **7.5. Training**

Here we will proceed with the fine-tuning training.

In [35]:

# ------------------------------------------------------------
# Fine-tuning using previously created chunks and training pairs
# ------------------------------------------------------------

from sentence_transformers import losses

output_path = "automotive_sentence_bert_finetuned"

# ------------------------------------------------------------
# Check required objects
# ------------------------------------------------------------

try:
    chunks
except NameError:
    raise RuntimeError(
        "Chunks were not found. Please execute the chunk creation section first."
    )

try:
    train_examples
except NameError:
    raise RuntimeError(
        "Training pairs were not found. Please execute the training pair creation section first."
    )

try:
    train_dataloader
except NameError:
    raise RuntimeError(
        "train_dataloader was not found. Please create it before fine-tuning."
    )

# ------------------------------------------------------------
# Preserve base embeddings before fine-tuning
# ------------------------------------------------------------

try:
    base_embeddings
    print("Using stored base embeddings.")
except NameError:
    try:
        chunk_embeddings
        base_embeddings = chunk_embeddings.clone()
        print("Stored current chunk embeddings as base embeddings.")
    except NameError:
        raise RuntimeError(
            "Base embeddings were not found. Please execute the embedding generation section first."
        )

# ------------------------------------------------------------
# Fine-tuning
# ------------------------------------------------------------

try:
    fine_tuning_completed
    print(
        "Fine-tuning already completed in this runtime. "
        "Skipping training."
    )

except NameError:

    train_loss = losses.MultipleNegativesRankingLoss(
        model
    )

    num_epochs = 1

    warmup_steps = int(
        len(train_dataloader)
        * num_epochs
        * 0.1
    )

    model.fit(
        train_objectives=[
            (train_dataloader, train_loss)
        ],
        epochs=num_epochs,
        warmup_steps=warmup_steps,
        show_progress_bar=True
    )

    fine_tuning_completed = True

    print("Fine-tuning completed.")

# ------------------------------------------------------------
# Save fine-tuned model
# ------------------------------------------------------------

model.save(output_path)

print(f"Fine-tuned model saved to: {output_path}")


Using stored base embeddings.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 0, 'pad_token_id': 1}.


Step,Training Loss


Fine-tuning completed.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned model saved to: automotive_sentence_bert_finetuned


## **7.6. Recreating embeddings**

Since the **weights of the BERT changed**, the embeddings produced for the chunks **already changed**.

Let's recreate these embeddings...


In [36]:
# ------------------------------------------------------------
# Recreate embeddings using the fine-tuned model
# ------------------------------------------------------------

texts = [
    chunk["text"]
    for chunk in chunks
]

ft_embeddings = model.encode(
    texts,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

# From this point on, the default retrieval functions use the fine-tuned
# embeddings, preserving the original notebook workflow.
chunk_embeddings = ft_embeddings

print("Updated embeddings generated with the fine-tuned model.")
print("Embedding matrix shape:", chunk_embeddings.shape)


Batches:   0%|          | 0/104 [00:00<?, ?it/s]

Updated embeddings generated with the fine-tuned model.
Embedding matrix shape: torch.Size([3327, 384])


## **7.7. Comparing models**

Now, let's compare the **fine-tuned model** with the **base model** without loading both models again.


In [37]:
# ------------------------------------------------------------
# Compare base and fine-tuned retrieval results
# ------------------------------------------------------------

queries = [
    "A luz do airbag ligou. Isso é normal?",
    "Consigo conectar som bluetooth no carro?",
    "O carro está derrapando quando freio",
]


def retrieval_summary_with_embeddings(
    model_to_use,
    embeddings,
    queries
):
    rows = []

    for query in queries:

        query_embedding = model_to_use.encode(
            query,
            convert_to_tensor=True,
            normalize_embeddings=True
        )

        scores = util.cos_sim(
            query_embedding,
            embeddings
        )[0]

        best_idx = int(scores.argmax())
        chunk = chunks[best_idx]

        if chunk["start_page"] == chunk["end_page"]:
            page_info = str(chunk["start_page"])
        else:
            page_info = f"{chunk['start_page']} - {chunk['end_page']}"

        rows.append({
            "Query": query,
            "Source file": chunk["source_file"],
            "Chunk": chunk["chunk_id"],
            "Pages": page_info,
            "Score": float(scores[best_idx])
        })

    return pd.DataFrame(rows)


# Generate summaries
base_summary = retrieval_summary_with_embeddings(
    model_to_use=model,
    embeddings=base_embeddings,
    queries=queries
)

ft_summary = retrieval_summary_with_embeddings(
    model_to_use=model,
    embeddings=ft_embeddings,
    queries=queries
)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("BASE MODEL RESULTS:")

display(
    base_summary.style
    .hide(axis="index")
    .background_gradient(
        cmap="Greens",
        subset=["Score"]
    )
    .set_properties(
        subset=["Query", "Source file"],
        **{"text-align": "left"}
    )
)

print("\nFINE-TUNED MODEL RESULTS:")

display(
    ft_summary.style
    .hide(axis="index")
    .background_gradient(
        cmap="Oranges",
        subset=["Score"]
    )
    .set_properties(
        subset=["Query", "Source file"],
        **{"text-align": "left"}
    )
)

BASE MODEL RESULTS:


Query,Source file,Chunk,Pages,Score
A luz do airbag ligou. Isso é normal?,Manual_VW_Saveiro-2027.pdf,25,44 - 46,0.755558
Consigo conectar som bluetooth no carro?,Manual-VW-Tiguan-2026.pdf,162,209 - 210,0.793051
O carro está derrapando quando freio,Manual-VW-Tiguan-2026.pdf,147,189 - 190,0.646669



FINE-TUNED MODEL RESULTS:


Query,Source file,Chunk,Pages,Score
A luz do airbag ligou. Isso é normal?,Manual_Chevrolet_Equinox_2024.pdf,45,95 - 98,0.811093
Consigo conectar som bluetooth no carro?,Manual_VW_Voyage_2023.pdf,96,144 - 146,0.732016
O carro está derrapando quando freio,Manual_Chevrolet_Captiva_2026.pdf,57,141 - 144,0.599048


Some highlights:

* The fine-tuned model retrieved different chunks for the queries, indicating that the **embedding space was successfully adapted** during training;
* The fine-tuning modified the geometry of the embedding space, making the cosine similarity values **not directly comparable** to those of the original model;
* The effectiveness of the fine-tuned model should be assessed primarily by the **relevance of the retrieved documents**, rather than by the magnitude of the similarity values alone.
<br>


Let's qualitatively compare the chunks retrieved by the base and fine-tuned models...

In [38]:
# ------------------------------------------------------------
# Compare retrieved chunks side by side
# ------------------------------------------------------------

import pandas as pd

def retrieve_best_chunk(
    query,
    model_to_use,
    embeddings
):
    query_embedding = model_to_use.encode(
        query,
        convert_to_tensor=True,
        normalize_embeddings=True
    )
    scores = util.cos_sim(
        query_embedding,
        embeddings
    )[0]

    best_idx = int(scores.argmax())
    return chunks[best_idx]

rows = []

for query in queries:
    base_chunk = retrieve_best_chunk(
        query,
        model,
        base_embeddings
    )
    ft_chunk = retrieve_best_chunk(
        query,
        model,
        ft_embeddings
    )
    rows.append({
        "Query": query,
        "Base Model":
            base_chunk["text"][:800] + "...",
        "Fine-Tuned Model":
            ft_chunk["text"][:800] + "..."
    })

comparison = pd.DataFrame(rows)

display(
    comparison.style
    .hide(axis="index")
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("font-weight", "bold")
            ]
        }
    ])
    .set_properties(
        subset=["Query"],
        **{
            "text-align": "left",
            "vertical-align": "top",
            "font-weight": "bold",
            "width": "260px"
        }
    )
    .set_properties(
        subset=["Base Model", "Fine-Tuned Model"],
        **{
            "text-align": "left",
            "vertical-align": "top",
            "width": "520px",
            "white-space": "normal"
        }
    )
)

Query,Base Model,Fine-Tuned Model
A luz do airbag ligou. Isso é normal?,"sistema de airbags e prétensio- nador do cinto de segurança. 42 Segurança Airbag frontal do passageiro dianteiro desligado A luz de c ontrole amarela se acende permanente- mente para o airbag frontal do passageiro dianteiro desligado. O airbag frontal do passageiro dianteiro foi desliga- do. Se, com o airbag frontal do passageiro desligado, a luz de controle PASSENGER AIR BAG não ac ender permanentemente ou se permanecer acesa juntamente com a luz de controle no instru- mento c ombinado, pode haver uma avaria no siste- ma de airbag → Página 40. — Verificar se o airbag frontal do passageiro diantei- ro deve permanecer desligado, por exemplo, ao utilizar uma cadeira de criança no banco do passa- geiro dianteiro. Transporte de crianças no veí- culo Introdução ao tema As c adeiras de criança r...","de segurança. Então cada luz pode acender ou piscar e um aviso sonoro será ativado caso o passageiro do banco traseiro não afivele ou desafivele o cinto de segurança, quando o veículo está em movimento. Se todos os cintos dos passageiros do banco traseiro estiverem afivelados, então o aviso sonoro não será ativado e as luzes não acenderão. Luz indicadora do airbag Esta luz indica que existe um problema elétrico com o sistema de airbag. Ela está localizada no conjunto de instrumentos. A verificação do sistema inclui o(s) sensor(es) do airbag, o sistema de detecção de passageiro, os pré- tensionadores, os módulos do airbag, a fiação e o módulo de detecção e diagnóstico de colisões. Para obter mais informações sobre o sistema de airbag, consulte Sistema de airbag 3 49. A luz indicadora do air..."
Consigo conectar som bluetooth no carro?,é possível em veí- culos que estão equipados com uma interface de te- lefone instalada de fábrica que suporta essa função. Perfil Bluetooth No máximo três dispositivos móveis podem estar conectados via Bluetooth ao mesmo tempo: dois pa- ra telefonia e um para reprodução de música. As seguintes funções de Bluetooth são suportadas: — Telefonia e viva-voz. — Reprodução de música. — Exibição e comando da reprodução de música. — Transmissão de Cover Arts. — Acesso a agenda e listas de chamadas. — Acesso a SMS e e-mail. Premissas para usar a interface Bluetooth para re- produção de música ✓ O dispositivo móvel suporta o perfil Bluetooth Advanced Audio Distribution Profile (A2DP). ✓ A transmissão de áudio e mídia para o sistema Infotainment deve estar permitida nas configu- rações do dispositivo ...,"fone com um telefone móvel Bluetooth® é neces- sário um único processo de pareamento. Alguns telefones móveis Bluetooth® são reco- nhecidos e conectados automaticamente ao ligar a ignição, se anteriormente já houve uma  Manual de instruções142 conexão. Ao mesmo tempo, o próprio telefone mó vel, bem como o Bluetooth® no telefone mó- vel, devem estar ligados e todas as conexões Bluetooth® ativas com outros aparelhos devem estar separadas. A comunicação Bluetooth® por rádio é gratuita. Bluetooth® é uma marca registrada da Blue- tooth® SIG, Inc. Perfis Bluetooth® Se um telefone móvel estiver conectado ao con- trole do telefone, o intercâmbio de dados será realizado por meio de um dos perfis Bluetooth®. Telefonia de base Bluetooth® Hands-Free-Profile (HFP): – Se um telefone móvel for conectado..."
O carro está derrapando quando freio,"obstáculo é detectado, o veículo freia até parar e é mantido parado por cerca de dois se- gundos. Assistente de estacionamento (Park Assist Plus): O veículo freia até parar e é retido pelo freio de esta- cionamento eletrônico. O processo de estaciona- mento é interrompido e deve ser reiniciado. Intervenção de frenagem automática pela função de frenagem de manobra. Imobilizar o veículo com o freio! Intervenção de frenagem automática do as- sistente de saída de v aga. Imobilizar o veículo com o freio! Dependendo da versão, uma mensagem de texto é exibida no sistema Infotainment ou no display do instrumento combinado digit

# **8. Semantic search vs fine-tuning approach**


| Aspect | Semantic Search | Fine-Tuning |
|----------|----------|----------|
| **Main Goal** | Retrieve relevant chunks | Improve retrieval quality |
| **Primary Component** | Encoder-only Transformer | Encoder-only Transformer |
| **Knowledge Location** | Knowledge Base | Knowledge Base + Model Weights |
| **Explicit Knowledge** | Knowledge Base | Knowledge Base |
| **Implicit Knowledge** | Pre-trained encoder | Domain-adapted encoder |
| **External Documents Needed?** | Yes | Yes |
| **Uses Knowledge Base?** | Yes | Yes |
| **Update Weights?** | No | Yes |
| **Can Replace Knowledge Base?** | No | No |
| **Output** | Ranked chunks | Improved embeddings |
| **Main Benefit** | Semantic retrieval | Better domain-specific retrieval |
| **Knowledge Freshness** | High | High |
| **Training Cost** | None | Medium |
| **Inference Cost** | Low | Low |
| **Hallucination Risk** | None | None |
| **Typical Models** | Sentence-BERT, E5, BGE | Fine-tuned Sentence-BERT |

<br>

# **9. Other Encoder-only Transformers**

BERT is the baseline encoder-only Transformer (also known as a 1st generation encoder). <br>
Some other avaliable options are: <br>
* Robustly Optimized BERT Approach (RoBERTa)
* Decoding-enhanced BERT with Disentagled Attention (DeBERTa)
* ...
<br>

<img src="https://drive.google.com/uc?export=view&id=1w93uH55HwYl3TgNyRMs5sDHxcWc1q90e" width="900">

____
# **PART II - Decoder-Only Transformers**
____


# **10. Overview**

* An encoder-only Transformer is limited to understanding and retrieving semantically relevant information from the Knowledge Base; **it cannot generate new text or produce complete natural-language responses**.
* A decoder-only Transformer overcomes this limitation by generating coherent, **context-aware responses based on the user's query and the retrieved manual sections**.
* In an automotive troubleshooting assistant, it can explain possible causes of a fault, recommend diagnostic procedures, summarize maintenance instructions, and provide step-by-step guidance in natural language.
* When integrated into a **Retrieval-Augmented Generation (RAG)** pipeline, the decoder-only model uses the retrieved chunks as grounding evidence, producing informative responses while reducing reliance on its internal knowledge alone.
* This combination enables the system to move beyond document retrieval, transforming relevant manual excerpts into **complete, human-readable troubleshooting assistance**.




Some of the available decoder-only Transformers are:

<img src="https://drive.google.com/uc?export=view&id=1XYbmsEfeXu7ADZA9dkHkoDb7uVY6xCF9" width="900">

# **11. Qwen**

In our implementation, let's adopt the **Tongyi Qianwen (Qwen)** model:

* 通义 (Tongyi) → "compreensão universal", "entendimento geral";
* 千问 (Qianwen) → "mil perguntas".

Here we will adopt a "Qwen/Qwen2.5-1.5B-Instruct", with 1.5 billion parameters.

## **11.1. Loading Qwen**

First of all, let's create the Qwen.

In [39]:
# ------------------------------------------------------------
# Load decoder-only Transformer (Qwen)
# ------------------------------------------------------------

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

!pip -q install transformers accelerate bitsandbytes
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline
)

try:
    generator
    print("Using previously loaded decoder-only model.")
except NameError:
    print("Loading decoder-only model...")
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME
    )

    decoder_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype="auto",
        device_map="auto"
    )

    generator = pipeline(
        "text-generation",
        model=decoder_model,
        tokenizer=tokenizer
    )

    print("Decoder-only model loaded.")

print(f"Model: {MODEL_NAME}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.4 MB/s eta 0:00:00
Loading decoder-only model...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Decoder-only model loaded.
Model: Qwen/Qwen2.5-1.5B-Instruct


## **11.2. Prompt construction**

First of all, let's **construct our prompt**..

In [40]:
# ------------------------------------------------------------
# Prompt construction
# ------------------------------------------------------------

def build_prompt(
    query,
    retrieved_chunks,
    max_chars_per_chunk=1500
):

    context = ""

    for i, chunk in enumerate(retrieved_chunks, start=1):

        context += (
            f"\n==============================\n"
            f"Retrieved Manual Section {i}\n"
            f"==============================\n"
        )

        context += chunk["text"][:max_chars_per_chunk]
        context += "\n\n"

    prompt = f"""
Use ONLY the information contained in the retrieved manual sections.
If the manuals do not provide enough information, explicitly state that.
Do not invent procedures.
Question:
{query}
Retrieved Manual Sections:
{context}
Answer:
"""

    return prompt

Let's create a function that will **display the contets** of the **initial prompt**...

In [41]:
# ------------------------------------------------------------
# Inspect initial prompt processing
# ------------------------------------------------------------

import pandas as pd

def inspect_initial_prompt(
    query,
    top_k=3,
    max_chars_per_chunk=1500,
    num_tokens_to_show=50
):

    retrieved_chunks = retrieve_relevant_chunks(
        query=query,
        top_k=top_k
    )

    prompt = build_prompt(
        query=query,
        retrieved_chunks=retrieved_chunks,
        max_chars_per_chunk=max_chars_per_chunk
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You are an automotive troubleshooting assistant. "
                "Use only the retrieved manual sections as evidence."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    encoded = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    )

    input_ids = encoded["input_ids"][0]

    tokens = tokenizer.convert_ids_to_tokens(
        input_ids
    )

    # --------------------------------------------------------
    # Prompt statistics
    # --------------------------------------------------------

    print("\nPrompt statistics:")

    print(f"Characters           : {len(formatted_prompt):,}")
    print(f"Input tokens         : {len(tokens):,}")
    print(f"Retrieved chunks     : {len(retrieved_chunks)}")
    print(f"Maximum new tokens   : {250:,}")   # ou max_new_tokens
    print(f"Estimated total size : {len(tokens) + 250:,} tokens")
    print("\n")

    # --------------------------------------------------------
    # Prompt inspection
    # --------------------------------------------------------

    print("=" * 100)
    print("INITIAL PROMPT INSPECTION")
    print("=" * 100)

    #print("\nPrompt size:")
    #print(f"Characters : {len(formatted_prompt)}")
    #print(f"Tokens     : {len(tokens)}")

    print("\nPrompt preview:\n")
    print(prompt[:3000])

    print("\nRetrieved chunks used in the prompt:")

    for i, chunk in enumerate(retrieved_chunks, start=1):

        if chunk["start_page"] == chunk["end_page"]:
            pages = str(chunk["start_page"])
        else:
            pages = f"{chunk['start_page']} - {chunk['end_page']}"

        print(
            f"{i}. {chunk['source_file']} | "
            f"Chunk {chunk['chunk_id']} | "
            f"Pages {pages} | "
            f"Similarity {chunk['score']:.3f}"
        )

    return {
        "formatted_prompt": formatted_prompt,
        "input_ids": input_ids,
        "tokens": tokens,
        "retrieved_chunks": retrieved_chunks
    }

## **11.3. Retrieval-Augmented Generation (RAG)**

Now, let's create a **function** implement our **Retrieval-Augmented Generation (RAG)**.

Our RAG will work as follows:

1. The user's query is first encoded into a **semantic embedding** using the fine-tuned Sentence-BERT encoder.
2. The query embedding is **compared with the embeddings of all knowledge-base chunks** using cosine similarity to identify the most semantically relevant information.
3. The **top-ranked chunks are retrieved and incorporated into a structured prompt**, providing contextual evidence for the generative model.
4. The prompt is then submitted to a decoder-only Transformer (Qwen), which **synthesizes a coherent natural-language response** based exclusively on the retrieved manual sections.
5. By grounding the generation process on retrieved documentation, the RAG architecture produces **context-aware automotive troubleshooting guidance** while reducing the likelihood of hallucinations.


In [42]:
# ------------------------------------------------------------
# Retrieval-Augmented Generation
# ------------------------------------------------------------

from tqdm.auto import tqdm
from transformers import TextIteratorStreamer
from threading import Thread
import torch
import textwrap

def automotive_rag(
    query,
    top_k=3,
    max_new_tokens=250,
    max_chars_per_chunk=1500,
    line_width=100
):

    progress = tqdm(
        total=4,
        desc="Starting RAG pipeline"
    )

    progress.set_description("Retrieving relevant chunks")

    retrieved_chunks = retrieve_relevant_chunks(
        query=query,
        top_k=top_k
    )

    progress.update(1)

    progress.set_description("Building prompt")

    prompt = build_prompt(
        query=query,
        retrieved_chunks=retrieved_chunks,
        max_chars_per_chunk=max_chars_per_chunk
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You are an automotive troubleshooting assistant. "
                "Use only the retrieved manual sections as evidence."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(decoder_model.device)

    progress.update(1)

    progress.set_description("Generating answer with Qwen")

    print("\nGENERATED ANSWER:\n")

    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True
    )

    generation_kwargs = {
        **inputs,
        "streamer": streamer,
        "max_new_tokens": max_new_tokens,
        "do_sample": False,
        "pad_token_id": tokenizer.eos_token_id
    }

    thread = Thread(
        target=decoder_model.generate,
        kwargs=generation_kwargs
    )

    thread.start()

    generated_text = ""
    current_line = ""

    for new_text in streamer:

        generated_text += new_text
        current_line += new_text

        while len(current_line) >= line_width:

            cut_position = current_line.rfind(
                " ",
                0,
                line_width
            )

            if cut_position == -1:
                cut_position = line_width

            print(
                current_line[:cut_position]
            )

            current_line = current_line[
                cut_position:
            ].lstrip()

    if current_line.strip():
        print(current_line.strip())

    thread.join()

    answer = generated_text.strip()

    progress.update(1)

    progress.set_description("Formatting results")

    print("\n")
    print("=" * 100)
    print("RETRIEVED MANUAL SECTIONS")
    print("=" * 100)

    for i, chunk in enumerate(retrieved_chunks, start=1):

        print("-" * 100)

        if chunk["start_page"] == chunk["end_page"]:
            pages = str(chunk["start_page"])
        else:
            pages = f"{chunk['start_page']} - {chunk['end_page']}"

        print(
            f"{i}. "
            f"{chunk['source_file']} | "
            f"Chunk {chunk['chunk_id']} | "
            f"Pages {pages} | "
            f"Similarity {chunk['score']:.3f}"
        )

    progress.update(1)
    progress.close()

    return answer

## **11.4. First prompt inspection**

The **initial prompt** will be composed as follows:

In [43]:
# ------------------------------------------------------------
# Inspect prompt before running RAG
# ------------------------------------------------------------

query = """
A luz do airbag ligou.
"""

inspection = inspect_initial_prompt(
    query=query,
    top_k=2,
    max_chars_per_chunk=1000,
    num_tokens_to_show=50
)


Prompt statistics:
Characters           : 2,604
Input tokens         : 659
Retrieved chunks     : 2
Maximum new tokens   : 250
Estimated total size : 909 tokens


INITIAL PROMPT INSPECTION

Prompt preview:


Use ONLY the information contained in the retrieved manual sections.
If the manuals do not provide enough information, explicitly state that.
Do not invent procedures.
Question:

A luz do airbag ligou.

Retrieved Manual Sections:

Retrieved Manual Section 1
veículo, a luz de adver- tência e o alerta sonoro do cinto de segurança pode não estar disponível. A luz de controle amarela no display do ins- trumento combinado se acende brevemente após ligar a ignição para o teste de funciona- mento e se apaga após alguns segundos. Airbag frontal do passageiro dianteiro desligado. A luz de c ontrole amarela no console central está acesa permanentemente → Fig. 31. Se, com o airbag frontal do passageiro desligado, a luz de controle PASSENGER AIR BAG não ac ender permanentemente ou se permanec

This prompt serves as the **interface between the retrieval and generation stages**, transforming the retrieved knowledge into contextual evidence that guides the decoder in producing a grounded natural-language answer.

Unlike the encoder, which receives only the user's query, the decoder receives a much richer input consisting of:

1. behavioral instructions,
2. the user's question, and
3. the retrieved manual sections.

This prompt provides all the **contextual information** required for the decoder to synthesize an evidence-based response.

## **11.5. Running the RAG**
Now let's run the RAG...

In [44]:
# ------------------------------------------------------------
# Execute main RAG
# ------------------------------------------------------------

answer = automotive_rag(
    query=query,              # User question submitted to the RAG system.
    top_k=2,                  # Number of most relevant chunks retrieved using semantic search
    max_new_tokens=150,       # Maximum number of tokens that the decoder-only Transformer is allowed to generate
    max_chars_per_chunk=1000  # Maximum number of characters extracted from each retrieved chunk and included in the prompt
)

Starting RAG pipeline:   0%|          | 0/4 [00:00<?, ?it/s]


GENERATED ANSWER:

The light on the airbag indicator has come on. This could indicate a problem with the airbag
system. The owner should immediately check and possibly repair or replace any faulty components
such as sensors for the airbags, collision detection systems, pre-tensioners, airbag modules,
wiring, and collision detection diagnostic module. It is important to ensure all seat belts of
passengers in the rear seats are fastened before starting the vehicle to avoid false alarms from
the airbag indicator light. If the light does not illuminate at startup, it should be checked and
fixed promptly.


RETRIEVED MANUAL SECTIONS
----------------------------------------------------------------------------------------------------
1. Manual_VW_Saveiro-2027.pdf | Chunk 23 | Pages 42 - 44 | Similarity 0.780
----------------------------------------------------------------------------------------------------
2. Manual_Chevrolet_Equinox_2024.pdf | Chunk 45 | Pages 95 - 98 | Similarity 0.777


After processing the **initial prompt**, the decoder generates the **first output token**.

This token is then **appended** to the existing input sequence and becomes part of the context for the **next forward pass**. This process is repeated **autoregressivelly**, with each newly generated token extending the context used to predict the next token, until the complete response is produced or the maximum number of tokens is reached.

Initial Prompt<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;↓ <br>
Prompt + Token1 <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;↓ <br>
Prompt + Token1 + Token2 <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;↓ <br>
Prompt + Token1 + Token2 + Token3 <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;↓ <br>
... <br>

# **12. Encoder-only vs Decoder-Only Transformers**

## **12.1. Overview**

It is important to note that the decoder:

* Is initialized with a **structured prompt** that combines instructions, the user's query, and retrieved information
* Uses the retrieved manual sections as **contextual evidence**
* Generates one token at a time (**autoregressive generation**)
* **Extends the prompt** after each generated token.
* Stops when the response is complete or the maximum number of tokens is reached.

Also, the **decoder does not**:

* Perform semantic search
* Calculate similarity between embeddings
* Choose which chunks to retrieve
* Directly query the knowledge base.

We can also observe that the decoder is **significantly slower** than the encoder during inference due to fundamental architectural differences:

* **Encoder-only:** performs a single forward pass to encode the input into a semantic embedding.

* **Decoder-only:** performs an initial forward pass to encode the context, followed by one additional forward pass for each generated token. Because text is generated autoregressively, decoder-only models require many sequential forward passes, resulting in substantially longer inference times.

## **12.2. Fine-Tuning**

* Fine-tuning is also possible in decoders, and represents **one of the most active areas of research in LLMs**!
* Unlike encoder, decoder fine-tuning **focuses on improving text generation** rather than semantic retrieval
* Unlike encoders, decoders training is typically performed using **supervised input-output pairs**:
   1. Model receives a *full prompt* (instructions + query + retrieved information)
   2. Model generates a *token*
   3. The token is compared to a *expected token*
   4. A cross-entropy loss measures the error between *predicted* and *expected tokens*
   5. Decoder weights are tuned based on **backpropagation**
   6. This process is repeated *thousand of times*
* Fine-tuning improves instruction following, domain-specific terminology, response quality, and reasoning within the target application
* In automotive troubleshooting, decoder fine-tuning would enables the model to produce **more reliable, consistent, and context-aware responses** based on the retrieved manual sections


<img src="https://drive.google.com/uc?export=view&id=1iWy1jX7M539LRUF2GPlA16vFylk3x3-W" width="900">


____
# **PART III - Encoder-Decoder Transformers**
___

# **13. Overview**

* Encoder–decoder architectures integrate understanding and generation into a **single model**, enabling end-to-end training but offering limited flexibility for independently improving retrieval and generation
* The modular encoder-only + decoder-only approach separates semantic retrieval from text generation, **allowing each component to be optimized**, fine-tuned, and replaced independently
* This modular design is particularly **well suited to RAG**, where external knowledge bases can be updated without retraining the generative model, providing greater scalability, flexibility, and maintainability for real-world applications.


<img src="https://drive.google.com/uc?export=view&id=1c3qdS3prpbPsoGIUwYlRr3GEUiGK-B3X" width="900">


# **14. Available models**

Here are some of the models currently available and their main characteristics...

<img src="https://drive.google.com/uc?export=view&id=1KhhMUPfuDzoAykLtyp4rVGAzq58iYP4z" width="900">



# **15. Next Steps**

You are now able to implement **Transformer-based solutions** for automotive troubleshooting, including:

- Semantic search with encoder-only models
- Domain adaptation through encoder fine-tuning
- Retrieval-Augmented Generation (RAG) using an encoder-only retriever and a decoder-only generator
- Conceptual comparison between encoder-only, decoder-only, and encoder-decoder Transformer architectures

As next steps, you can:

- Apply these techniques to other domains
- Experiment with more powerful encoders such as RoBERTa, DeBERTa, E5, BGE, or GTE
- Compare decoder-only models such as Qwen, Llama, Mistral, and DeepSeek
- Explore encoder-decoder models such as T5, BART, FLAN-T5, mT5, and UL2
- Fine-tune encoder-only models using domain-specific datasets
- Fine-tune decoder-only models for domain-specific response generation
- Fine-tune encoder-decoder models for sequence-to-sequence tasks such as summarization, translation, and question answering
- Analyze the trade-off between retrieval accuracy, generation quality, inference time, modularity, and hallucination risk

<br>
<br>
<br>


____

<center>
<img src="https://upload.wikimedia.org/wikipedia/commons/0/0a/Logo_Unesp.svg" width="400" style="float: left; margin-right: 5px;" border="0px" />
</center>